# Quadruplet Metric Learning + SVM

Trains a lightweight embedding network with a hard-mined quadruplet loss on top of the frozen DenseNet121+ResNet50 features, then classifies the learned embeddings with an SVM. Goal: improve separability of visually-similar / minority lesion classes beyond what the raw-feature SVM baseline achieves. See `src/models/quadruplet_net.py` for the cleaned, importable version of this pipeline, and the main README for headline results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
!pip install torch umap-learn scikit-learn tqdm seaborn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import WeightedRandomSampler


## 1. Setup
Device selection.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load extracted deep features
`features.npy` / `labels.npy` are produced by `src/features/feature_extraction.py` (DenseNet121 + ResNet50 concatenated features).

In [ ]:
X=np.load("features.npy")
y= np.load("labels.npy")

In [ ]:
print("Feature shape:", X.shape)
print("Label shape:", y.shape)

In [ ]:
#check class imbalance
np.bincount(y)

## 3. Class imbalance check & train/val split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 4. Embedding-space visualization helpers (t-SNE, UMAP, class-similarity heatmap)

In [ ]:
# -------------------------
# VISUALIZATION FUNCTIONS
# -------------------------

from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt

def plot_tsne(X, y, title):

    # use subset if dataset is large
    if len(X) > 3000:
        idx = np.random.choice(len(X), 3000, replace=False)
        X = X[idx]
        y = y[idx]

    tsne = TSNE(
        n_components=2,
        perplexity=40,
        learning_rate=200,
        init="pca",
        random_state=42
    )

    X_embedded = tsne.fit_transform(X)

    plt.figure(figsize=(6,6))
    plt.scatter(X_embedded[:,0], X_embedded[:,1], c=y, cmap="tab10", s=8)
    plt.title(title)
    plt.show()


def plot_umap(X, y, title):

    reducer = umap.UMAP()
    X_embedded = reducer.fit_transform(X)

    plt.figure(figsize=(6,6))
    plt.scatter(X_embedded[:,0], X_embedded[:,1], c=y, cmap="tab10")
    plt.title(title)
    plt.show()


def class_similarity(X, y):

    classes = np.unique(y)
    means = []

    for c in classes:
        means.append(X[y==c].mean(axis=0))

    means = np.stack(means)

    sim = np.corrcoef(means)

    plt.figure(figsize=(6,5))
    sns.heatmap(sim, annot=True, cmap="coolwarm")
    plt.title("Class Similarity Matrix")
    plt.show()


## 5. Baseline visualization — before any metric learning

In [ ]:
# -------------------------
# BEFORE TRAINING VISUALIZATION
# -------------------------

plot_tsne(X_train, y_train, "t-SNE BEFORE training")
plot_umap(X_train, y_train, "UMAP BEFORE training")
class_similarity(X_train, y_train)

## 6. Dataset / DataLoader

In [ ]:
# -------------------------
# DATASET
# -------------------------

class QuadDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        x = self.X[idx]
        label = self.y[idx]

        return x, label


train_dataset = QuadDataset(X_train, y_train)
val_dataset = QuadDataset(X_val, y_val)

class_counts = np.bincount(y_train)

# inverse frequency weights
class_weights = 1.0 / class_counts

# weight for each sample
sample_weights = class_weights[y_train]

sample_weights = torch.tensor(sample_weights, dtype=torch.double)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

In [ ]:
# VERIFY SAMPLING
# -------------------------

for X_batch, y_batch in train_loader:
    print("Batch class counts:", torch.bincount(y_batch))
    break

## 7. Quadruplet mining
For every anchor: hardest same-class positive, a semi-hard negative, and a second negative — see `src/models/quadruplet_net.py::mine_quadruplets` for the cleaned, documented version.

In [ ]:
def get_quadruplets(embeddings, labels, margin=1.0):

    dist = torch.cdist(embeddings, embeddings)

    anchors = []
    positives = []
    negatives1 = []
    negatives2 = []

    for i in range(len(labels)):

        anchor_label = labels[i]

        pos_mask = labels == anchor_label
        neg_mask = labels != anchor_label

        pos_indices = torch.where(pos_mask)[0]
        neg_indices = torch.where(neg_mask)[0]

        # remove self
        pos_indices = pos_indices[pos_indices != i]

        if len(pos_indices) == 0 or len(neg_indices) == 0:
            continue

        # distances
        pos_dist = dist[i][pos_indices]
        neg_dist = dist[i][neg_indices]

        # -------- Top-k Hard Positive --------
        k = min(3, len(pos_indices))
        topk = torch.topk(pos_dist, k).indices
        pos_candidates = pos_indices[topk]
        hard_pos = pos_indices[torch.argmax(pos_dist)]

        # -------- Semi-Hard Negative --------
        pos_max = pos_dist.max()

        mask = (neg_dist > pos_max) & (neg_dist < pos_max + margin)

        semi_hard = neg_indices[mask]

        if len(semi_hard) > 0:
            hard_neg1 = semi_hard[torch.randint(0, len(semi_hard), (1,))]
        else:
            # fallback to closest negative
            hard_neg1 = neg_indices[torch.argmin(neg_dist)]

        # second negative (for quadruplet)
        sorted_neg = neg_indices[torch.argsort(neg_dist)]
        hard_neg2 = sorted_neg[1] if len(sorted_neg) > 1 else sorted_neg[0]

        anchors.append(i)
        positives.append(hard_pos.item())
        negatives1.append(hard_neg1.item())
        negatives2.append(hard_neg2.item())

    return anchors, positives, negatives1, negatives2

## 8. Embedding network (small MLP head on top of the frozen deep features)

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )

    def forward(self, x):

        x = self.net(x)

        # no normalization
        return x


model = EmbeddingNet(X.shape[1]).to(device)

## 9. Quadruplet loss

In [ ]:

# -------------------------
# QUADRUPLET LOSS
# -------------------------

def quadruplet_loss(a,p,n1,n2):
    margin2 = 0.3
    margin1 = 1
    d_ap = F.pairwise_distance(a,p)
    d_an = F.pairwise_distance(a,n1)
    d_pn = F.pairwise_distance(p,n2)

    loss1 = F.relu(d_ap - d_an + margin1)
    loss2 = F.relu(d_ap - d_pn + margin2)

    return (loss1 + loss2).mean()


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

## 10. Training loop (100 epochs)

In [ ]:
epochs = 100

for epoch in range(epochs):

    model.train()
    
    total_loss = 0
    total_dap = 0
    total_dan = 0
    
    num_batches = 0
    dist_batches = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for X_batch, y_batch in progress_bar:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        embeddings = model(X_batch)

        a, p, n1, n2 = get_quadruplets(embeddings, y_batch)

        if len(a) == 0:
            continue

        anchor = embeddings[a]
        positive = embeddings[p]
        negative1 = embeddings[n1]
        negative2 = embeddings[n2]

        # distances
        d_ap = F.pairwise_distance(anchor, positive)
        d_an = F.pairwise_distance(anchor, negative1)

        # accumulate stats
        total_dap += d_ap.mean().item()
        total_dan += d_an.mean().item()
        dist_batches += 1

        loss = quadruplet_loss(anchor, positive, negative1, negative2)
        loss += 0.001 * torch.mean(torch.norm(embeddings, dim=1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        progress_bar.set_postfix(loss=loss.item())

    # averages per epoch
    avg_loss = total_loss / num_batches
    avg_dap = total_dap / dist_batches
    avg_dan = total_dan / dist_batches

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_loss:.4f}")
    print(f"Mean d(a,p): {avg_dap:.4f}")
    print(f"Mean d(a,n): {avg_dan:.4f}")

## 11. Embedding-space visualization — after metric learning
Compare against step 5 to see how much tighter/more separable the classes become.

In [ ]:
# -------------------------
# EMBEDDINGS AFTER TRAINING
# -------------------------

model.eval()

with torch.no_grad():
    embeddings = model(torch.tensor(X).float().to(device)).cpu().numpy()

# -------------------------
# AFTER TRAINING VISUALIZATION
# -------------------------

plot_tsne(embeddings, y, "t-SNE AFTER training")
plot_umap(embeddings, y, "UMAP AFTER training")
class_similarity(embeddings, y)

## 12. Extract embeddings for the train/val split

In [ ]:
import torch

model.eval()

def extract_embeddings(model, loader):
    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            embeddings = model(X_batch)

            all_embeddings.append(embeddings.cpu())
            all_labels.append(y_batch)

    embeddings = torch.cat(all_embeddings).numpy()
    labels = torch.cat(all_labels).numpy()

    return embeddings, labels


X_train_emb, y_train_emb = extract_embeddings(model, train_loader)
X_val_emb, y_val_emb = extract_embeddings(model, val_loader)

## 13. Train an SVM on the learned embeddings

In [ ]:

from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

svm.fit(X_train_emb, y_train_emb)

In [ ]:
y_pred = svm.predict(X_val_emb)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_val_emb, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val_emb, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

print("Accuracy:", accuracy_score(y_val_emb, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_val_emb, y_pred))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Macro F1:", f1_score(y_val_emb, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val_emb, y_pred, average="weighted"))

## 14. Final evaluation on the held-out test set

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# ==============================
# STEP 1: LOAD TEST DATA (FIXED)
# ==============================
X_test = np.load("test_features.npy")
y_test = np.load("test_labels.npy")

print("Test feature shape:", X_test.shape)
print("Test label shape:", y_test.shape)
print("Unique test labels:", np.unique(y_test))

# ==============================
# STEP 2: DATASET + LOADER
# ==============================
class NPYDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_loader = DataLoader(NPYDataset(X_test, y_test), batch_size=64, shuffle=False)

# ==============================
# STEP 3: DEVICE
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ==============================
# STEP 4: EXTRACT EMBEDDINGS
# ==============================
model.eval()
all_embeddings = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)

        emb = model(xb)   # embeddings

        all_embeddings.append(emb.cpu())
        all_labels.append(yb)

X_test_emb = torch.cat(all_embeddings).numpy()
y_test_emb = torch.cat(all_labels).numpy()

# ==============================
# STEP 5: USE SVM
# ==============================
y_pred = svm.predict(X_test_emb)

# ==============================
# STEP 6: REPORT
# ==============================
print("\nClassification Report (TEST):")
print(classification_report(y_test_emb, y_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_emb, y_pred))